In [ ]:
import geopandas as gpd

# Path to the geopackage
path = "data/lipas-points-espoo.gpkg"

# Read the specific layer
gdf = gpd.read_file(path, layer="lipas-points-espoo")

# Inspect the data
print(gdf.head())
print(gdf.columns)

# Interactive map
gdf.explore(
    tiles="CartoDB positron",
    tooltip=True,
    marker_kwds={"radius": 6},
)

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(r"data\baanat.zip")

print(gdf.head())
print(gdf.crs)

gdf.explore(
    tiles="CartoDB positron",
    tooltip=True,
)

In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd

place = "Espoo, Finland"

# Pull all drivable/cyclable street edges with useful tags
custom_filter = """
["highway"]
"""

tags = {
    "highway": True,
    "name": True,
    "cycleway": True,
    "cycleway:left": True,
    "cycleway:right": True,
    "cycleway:both": True,
    "bicycle": True,
    "oneway": True,
    "oneway:bicycle": True,
    "segregated": True,
    "surface": True,
    "smoothness": True,
    "maxspeed": True,
    "traffic_calming": True,
    "lit": True,
    "barrier": True,
    "access": True,
    "lanes": True
}

G = ox.graph_from_place(place, network_type="bike", simplify=True)
nodes, edges = ox.graph_to_gdfs(G)

# Keep relevant columns that actually exist
cols = [c for c in tags.keys() if c in edges.columns]
bike_edges = edges[cols + ["geometry"]].copy()

def classify_bike_facility(row):
    vals = {
        str(row.get("cycleway", "")).lower(),
        str(row.get("cycleway:left", "")).lower(),
        str(row.get("cycleway:right", "")).lower(),
        str(row.get("cycleway:both", "")).lower(),
    }

    if "track" in vals:
        return "protected_track"
    if "lane" in vals:
        return "painted_lane"
    if "shared_lane" in vals or "share_busway" in vals:
        return "shared_facility"

    maxspeed = str(row.get("maxspeed", "")).lower()
    calming = pd.notna(row.get("traffic_calming"))
    if calming:
        return "traffic_calmed_street"
    if maxspeed in {"20", "20 km/h", "30", "30 km/h"}:
        return "low_speed_street"

    return "mixed_traffic"

bike_edges["bike_class"] = bike_edges.apply(classify_bike_facility, axis=1)

print(bike_edges.head())

In [ ]:
m = bike_edges.explore(
    column="bike_class",
    cmap="Set2",
    linewidth=3,
    legend=True
)

nodes.explore(
    m=m,
    color="black",
    marker_kwds={"radius": 2}
)

In [ ]:
# from pathlib import Path

# out_path = Path("output/bike_network.gpkg")
# out_path.parent.mkdir(parents=True, exist_ok=True)

# bike_edges.reset_index().to_file(
#     out_path,
#     layer="edges",
#     driver="GPKG"
# )

# nodes.reset_index().to_file(
#     out_path,
#     layer="nodes",
#     driver="GPKG"
# )

In [ ]:
from pathlib import Path

import geopandas as gpd
import osmnx as ox


# ----------------------------
# Settings
# ----------------------------
place = "Espoo, Finland"
out_dir = Path("output/espoo_cycle_map")
out_dir.mkdir(parents=True, exist_ok=True)

bike_infra_fp = out_dir / "bike_infra.gpkg"


# ----------------------------
# 1) Download bikeable network
# ----------------------------
# OSMnx supports downloading bike networks directly with network_type="bike".
G = ox.graph_from_place(place, network_type="bike", simplify=True)
nodes, edges = ox.graph_to_gdfs(G)

# Reset index so osmid/u/v/key are regular columns before export
nodes = nodes.reset_index()
edges = edges.reset_index()

# Save graph layers
nodes.to_file(bike_infra_fp, layer="bike_nodes", driver="GPKG")
edges.to_file(bike_infra_fp, layer="bike_edges", driver="GPKG")

print(f"Saved nodes and edges: {bike_infra_fp}")

# ----------------------------
# 2) Download signed bicycle routes
# ----------------------------
# Signed bicycle route relations in OSM use:
# type=route, route=bicycle, network in {lcn, rcn, ncn, icn}
route_tags = {
    "route": "bicycle",
    "network": ["lcn", "rcn", "ncn", "icn"],
}

cycle_routes = ox.features_from_place(place, route_tags).reset_index()

# Optional: keep a useful subset of columns if present
route_cols_preferred = [
    "element",
    "id",
    "route",
    "network",
    "ref",
    "name",
    "operator",
    "type",
    "geometry",
]
route_cols = [c for c in route_cols_preferred if c in cycle_routes.columns]
if route_cols:
    cycle_routes_export = cycle_routes[route_cols].copy()
else:
    cycle_routes_export = cycle_routes.copy()

cycle_routes_export.to_file(
    bike_infra_fp,
    layer="cycle_routes",
    driver="GPKG",
)

print(f"Saved signed cycle routes: {bike_infra_fp}")
print(cycle_routes_export["network"].value_counts(dropna=False))


# ----------------------------
# 3) Download physical cycling infrastructure + amenities
# ----------------------------
# This captures mapped infrastructure and bike-related POIs.
infra_tags = {
    "highway": ["cycleway", "path"],
    "cycleway": True,
    "cycleway:left": True,
    "cycleway:right": True,
    "cycleway:both": True,
    "bicycle": True,
    "route": "bicycle",
    "amenity": ["bicycle_parking", "bicycle_repair_station", "bicycle_rental"],
    "shop": "bicycle",
}

bike_infra = ox.features_from_place(place, infra_tags).reset_index()

infra_cols_preferred = [
    "element",
    "id",
    "highway",
    "cycleway",
    "cycleway:left",
    "cycleway:right",
    "cycleway:both",
    "bicycle",
    "route",
    "network",
    "amenity",
    "shop",
    "name",
    "ref",
    "geometry",
]
infra_cols = [c for c in infra_cols_preferred if c in bike_infra.columns]
if infra_cols:
    bike_infra_export = bike_infra[infra_cols].copy()
else:
    bike_infra_export = bike_infra.copy()

bike_infra_export.to_file(
    bike_infra_fp,
    layer="bike_infra_ext",
    driver="GPKG",
)

print(f"Saved bike infrastructure: {bike_infra_fp}")


# ----------------------------
# 4) Quick summary
# ----------------------------
print("\nSummary")
print("-------")
print(f"Bike network nodes: {len(nodes):,}")
print(f"Bike network edges: {len(edges):,}")
print(f"Signed cycle-route features: {len(cycle_routes_export):,}")
print(f"Bike infrastructure features: {len(bike_infra_export):,}")

if "geom_type" not in bike_infra_export.columns:
    bike_infra_export["geom_type"] = bike_infra_export.geometry.geom_type
print("\nInfrastructure geometry types:")
print(bike_infra_export["geom_type"].value_counts(dropna=False))

In [ ]:
place = "Espoo, Finland"

route_tags = {
    "type": "route",
    "route": "bicycle",
    "network": ["lcn", "rcn", "ncn", "icn"],
}

cycle_routes = ox.features_from_place(place, route_tags)

In [ ]:
len(cycle_routes)

In [ ]:
place = "Espoo, Finland"

tags = {
    "type": "route",
    "route": "bicycle",
    "network": "ncn",
    "ref": True
}

gdf_routes = ox.features_from_place(place, tags)

In [ ]:
routes = gdf_routes[gdf_routes.geometry.type.isin(["LineString", "MultiLineString"])].copy()

In [ ]:
routes.explore()